# WaterSoftHack  
## Cloud Computing Hands-On with NRP Nautilus  
### Notebook 02 — ACCESS IDs, Credits, and Running a Real Cloud Job

Welcome back.

The first notebook checked that your Nautilus/Jupyter environment works.  
This notebook is more hands-on: you will connect your **ACCESS identity**, your **Nautilus namespace**, and a small **cloud workload** that can run as a Kubernetes Job.

This is the practical cloud lesson:

> Cloud computing means you do not just run code. You request remote resources, attach them to an identity/project, run a workload, observe it, and clean it up.

---

## What you will do

By the end of this notebook, you should be able to:

- record your ACCESS username / project information for the workshop
- verify whether `kubectl` can see Nautilus
- identify your active Kubernetes namespace
- build a tiny batch workload that does real work
- tag that workload with your workshop metadata
- submit the workload to Nautilus when instructed
- read job status and logs
- estimate the resource request behind the job
- clean up the job so you do not waste shared resources
- connect the hands-on job to ACCESS credit/usage thinking

---

## Instructor note

This notebook is safe to run in two modes:

1. **Dry-run mode**: students generate the workload and manifest locally, but do not submit anything.
2. **Live Nautilus mode**: students set `RUN_REAL_NAUTILUS_JOB = True` and submit the small Kubernetes Job.

The default is dry-run mode so nobody accidentally consumes allocation resources before the instructor is ready.


# 1. Enter your workshop identity information

Fill in the variables below.

Use:

- your **ACCESS username / ACCESS ID**
- your **ACCESS project or allocation identifier**, if your instructor gave one
- your **Nautilus namespace**

Do **not** paste passwords, tokens, private keys, kubeconfig contents, or secret credentials into this notebook.


In [ ]:
from pathlib import Path
import os
import re
import json
import time
import shutil
import subprocess
from datetime import datetime, timezone

# -------------------------------------------------------------------
# EDIT THESE THREE VALUES FOR THE WORKSHOP
# -------------------------------------------------------------------
ACCESS_USERNAME = os.environ.get("ACCESS_USERNAME", "replace-with-your-access-username")
ACCESS_PROJECT_ID = os.environ.get("ACCESS_PROJECT_ID", "replace-with-your-project-or-allocation-id")
NAUTILUS_NAMESPACE = os.environ.get("NAUTILUS_NAMESPACE", "replace-with-your-nautilus-namespace")

# Dry-run by default. Set to True only when the instructor says to launch the job.
RUN_REAL_NAUTILUS_JOB = False

# If you run a real job, keep this True for a workshop so resources are cleaned up after logs are collected.
DELETE_JOB_AFTER_LOGS = True

workspace = Path.home() / "watersofthack_cloud_nautilus" / "notebook_02_access_credits_job"
workspace.mkdir(parents=True, exist_ok=True)

print("Workspace:", workspace)
print("ACCESS_USERNAME:", ACCESS_USERNAME)
print("ACCESS_PROJECT_ID:", ACCESS_PROJECT_ID)
print("NAUTILUS_NAMESPACE:", NAUTILUS_NAMESPACE)
print("RUN_REAL_NAUTILUS_JOB:", RUN_REAL_NAUTILUS_JOB)


## Why do we record these values?

In cloud/HPC platforms, compute is usually connected to:

- **who you are**
- **what project/allocation you belong to**
- **what namespace or account is allowed to run workloads**
- **what resources your workload asks for**

This notebook writes those values into local metadata and, when running a job, into Kubernetes labels/annotations so your work is easier to identify.


# 2. Helper functions

The next cell defines safe command helpers.  
The helpers print command output without crashing the whole notebook when a tool is missing.


In [ ]:
def run_command(command, *, check=False, capture=True, timeout=60):
    # Run a shell command and return a dictionary with stdout/stderr/returncode.
    if isinstance(command, str):
        shell = True
        display_command = command
    else:
        shell = False
        display_command = " ".join(str(x) for x in command)

    print(f"$ {display_command}")
    try:
        result = subprocess.run(
            command,
            shell=shell,
            text=True,
            capture_output=capture,
            timeout=timeout,
            check=check,
        )
        if capture:
            if result.stdout:
                print(result.stdout.rstrip())
            if result.stderr:
                print(result.stderr.rstrip())
        return {
            "ok": result.returncode == 0,
            "returncode": result.returncode,
            "stdout": result.stdout if capture else "",
            "stderr": result.stderr if capture else "",
            "command": display_command,
        }
    except Exception as e:
        print(f"Command failed before completion: {type(e).__name__}: {e}")
        return {
            "ok": False,
            "returncode": None,
            "stdout": "",
            "stderr": str(e),
            "command": display_command,
        }


def sanitize_label_value(value, fallback="unknown"):
    # Make a Kubernetes label-compatible value.
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9_.-]+", "-", value)
    value = value.strip("-_.")
    if not value:
        value = fallback
    return value[:63]


def placeholder(value):
    return str(value).startswith("replace-with-") or str(value).strip() == ""


# 3. Check this Jupyter environment again

This confirms where the notebook kernel is running.  
It should be a remote Nautilus/Jupyter environment for the live workshop.


In [ ]:
import platform
import sys

print("Python:", sys.version.splitlines()[0])
print("Hostname:", platform.node())
print("Platform:", platform.platform())
print("User:", os.environ.get("USER", "(unknown)"))
print("HOME:", Path.home())
print("Current directory:", Path.cwd())


# 4. Check Kubernetes tooling

Nautilus is Kubernetes-based, so the hands-on cloud workload uses `kubectl`.

Run the next cell and read the output carefully.

If `kubectl` is missing or not authenticated yet, this notebook can still generate the manifest, but it cannot submit the live job.


In [ ]:
kubectl_path = shutil.which("kubectl")
print("kubectl path:", kubectl_path or "not found")

checks = {}
checks["kubectl_path"] = kubectl_path
checks["kubectl_version"] = run_command(["kubectl", "version", "--client"], timeout=30) if kubectl_path else {"ok": False, "stdout": "", "stderr": "kubectl not found"}
checks["current_context"] = run_command(["kubectl", "config", "current-context"], timeout=30) if kubectl_path else {"ok": False, "stdout": "", "stderr": "kubectl not found"}
checks["configured_namespace"] = run_command(["kubectl", "config", "view", "--minify", "-o", "jsonpath={..namespace}"], timeout=30) if kubectl_path else {"ok": False, "stdout": "", "stderr": "kubectl not found"}


## Choose the namespace for this notebook

If your kubeconfig already has a default namespace, the notebook can infer it.  
If not, it uses `NAUTILUS_NAMESPACE` from the first configuration cell.


In [ ]:
inferred_namespace = ""
if checks.get("configured_namespace", {}).get("ok"):
    inferred_namespace = checks["configured_namespace"].get("stdout", "").strip()

if placeholder(NAUTILUS_NAMESPACE) and inferred_namespace:
    active_namespace = inferred_namespace
else:
    active_namespace = NAUTILUS_NAMESPACE

print("Namespace inferred from kubeconfig:", inferred_namespace or "(none)")
print("Namespace selected for this notebook:", active_namespace)

if placeholder(active_namespace):
    print("\nACTION NEEDED: set NAUTILUS_NAMESPACE in Cell 1 before submitting a real job.")


# 5. Build a workshop identity card

This is a small local record linking your notebook, ACCESS identity, project/allocation, namespace, and run mode.

It does not contact ACCESS. It just helps keep your work organized.


In [ ]:
identity_card = {
    "workshop": "WaterSoftHack Cloud Computing",
    "notebook": "02 - ACCESS IDs, Credits, and Nautilus Jobs",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "access_username": ACCESS_USERNAME,
    "access_project_id": ACCESS_PROJECT_ID,
    "nautilus_namespace": active_namespace,
    "run_real_nautilus_job": RUN_REAL_NAUTILUS_JOB,
    "kubectl_found": kubectl_path is not None,
    "kubectl_context": checks.get("current_context", {}).get("stdout", "").strip(),
}

identity_path = workspace / "workshop_identity_card.json"
identity_path.write_text(json.dumps(identity_card, indent=2), encoding="utf-8")
print(identity_path.read_text(encoding="utf-8"))


# 6. Cloud allocation thinking: credits are connected to requested resources

ACCESS uses a credit/allocation model. In this workshop, the important habit is to connect a workload with the resources it asks for.

For this small job we use **CPU only**:

- no GPU
- small CPU request
- small memory request
- short-running command that exits by itself

That is intentional. Shared systems reward good citizenship: request what you need, run the work, collect output, clean up.


# 7. Plan a tiny cloud workload

The job will simulate a small water-sensor processing task:

1. generate synthetic readings
2. aggregate flow by region
3. print a JSON summary to stdout
4. exit cleanly

This gives us something real to submit, monitor, and collect logs from without using a large amount of shared infrastructure.


In [ ]:
# Workload knobs. Keep these small for a classroom workshop.
WORK_UNITS = 25_000
WATER_REGION = "demo-basin"

# Kubernetes resource requests/limits for the job.
# NRP policy recommends setting requests carefully and limits near requests.
resource_plan = {
    "requests": {
        "cpu": "250m",
        "memory": "256Mi",
        "ephemeral-storage": "512Mi",
    },
    "limits": {
        "cpu": "300m",
        "memory": "300Mi",
        "ephemeral-storage": "600Mi",
    },
}

resource_plan_path = workspace / "resource_plan.json"
resource_plan_path.write_text(json.dumps(resource_plan, indent=2), encoding="utf-8")
print(resource_plan_path.read_text(encoding="utf-8"))


## Estimate the request footprint

This is not an official ACCESS credit calculation. It is a workshop estimate to help you think like a cloud user.

The official usage/balance view is checked in the ACCESS allocations portal or the resource provider dashboards. Here we simply estimate what we asked Kubernetes to reserve.


In [ ]:
def parse_cpu_to_cores(cpu):
    cpu = str(cpu).strip()
    if cpu.endswith("m"):
        return float(cpu[:-1]) / 1000.0
    return float(cpu)


def parse_memory_to_gib(mem):
    mem = str(mem).strip()
    units = {
        "Ki": 1 / (1024 * 1024),
        "Mi": 1 / 1024,
        "Gi": 1,
        "Ti": 1024,
    }
    for suffix, factor in units.items():
        if mem.endswith(suffix):
            return float(mem[:-len(suffix)]) * factor
    # Assume bytes if no suffix.
    return float(mem) / (1024 ** 3)

assumed_runtime_minutes = 2
requested_cpu_cores = parse_cpu_to_cores(resource_plan["requests"]["cpu"])
requested_memory_gib = parse_memory_to_gib(resource_plan["requests"]["memory"])

footprint = {
    "assumed_runtime_minutes": assumed_runtime_minutes,
    "requested_cpu_cores": requested_cpu_cores,
    "requested_memory_gib": requested_memory_gib,
    "estimated_cpu_core_hours": requested_cpu_cores * assumed_runtime_minutes / 60,
    "estimated_memory_gib_hours": requested_memory_gib * assumed_runtime_minutes / 60,
}

footprint_path = workspace / "planned_workload_footprint.json"
footprint_path.write_text(json.dumps(footprint, indent=2), encoding="utf-8")
print(footprint_path.read_text(encoding="utf-8"))


# 8. Create the code that the cloud job will run

The actual Kubernetes Job will run a small Python program inside a container.

The program is embedded directly in the Job manifest so you do not need to build a Docker image for this lesson.


In [ ]:
job_python_code = r'''
import json
import math
import os
import random
import statistics
import time
from datetime import datetime, timezone

access_username = os.environ.get("ACCESS_USERNAME", "unknown")
access_project_id = os.environ.get("ACCESS_PROJECT_ID", "unknown")
region = os.environ.get("WATER_REGION", "demo-basin")
work_units = int(os.environ.get("WORK_UNITS", "25000"))

random.seed(42)
start = time.time()

stations = ["upstream", "midstream", "downstream", "reservoir"]
readings = {station: [] for station in stations}

# Meaningful but small computation: simulate flow readings and aggregate them.
for i in range(work_units):
    station = stations[i % len(stations)]
    seasonal = 3.0 * math.sin(i / 997.0)
    storm_pulse = 8.0 if i % 7000 < 350 else 0.0
    noise = random.random() * 1.5
    baseline = 12.0 + stations.index(station) * 2.7
    flow_cms = baseline + seasonal + storm_pulse + noise
    readings[station].append(flow_cms)

summary = {
    "workshop": "WaterSoftHack",
    "region": region,
    "access_username": access_username,
    "access_project_id": access_project_id,
    "work_units": work_units,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "runtime_seconds": round(time.time() - start, 4),
    "stations": {
        station: {
            "count": len(values),
            "mean_flow_cms": round(statistics.mean(values), 3),
            "max_flow_cms": round(max(values), 3),
        }
        for station, values in readings.items()
    },
}

print("WATERSOFTHACK_JOB_RESULT_START")
print(json.dumps(summary, indent=2, sort_keys=True))
print("WATERSOFTHACK_JOB_RESULT_END")
'''.strip()

script_path = workspace / "watersofthack_cloud_job.py"
script_path.write_text(job_python_code + "\n", encoding="utf-8")
print("Saved local copy of job script:", script_path)
print(script_path.read_text(encoding="utf-8")[:1200] + "\n...")


# 9. Test the job code locally inside this notebook

Before submitting work to a cloud platform, test the logic locally in the notebook environment.

This is a good cloud habit: debug cheaply before scheduling remote resources.


In [ ]:
local_test_env = os.environ.copy()
local_test_env.update({
    "ACCESS_USERNAME": ACCESS_USERNAME,
    "ACCESS_PROJECT_ID": ACCESS_PROJECT_ID,
    "WATER_REGION": WATER_REGION,
    "WORK_UNITS": "1000",  # smaller local smoke test
})

local_test = subprocess.run(
    [sys.executable if 'sys' in globals() else 'python', str(script_path)],
    text=True,
    capture_output=True,
    env=local_test_env,
    timeout=60,
)

print("Return code:", local_test.returncode)
print(local_test.stdout)
if local_test.stderr:
    print("STDERR:")
    print(local_test.stderr)


# 10. Build the Kubernetes Job manifest

The manifest is the cloud contract:

- what container image to run
- what command to execute
- what namespace to run in
- what CPU/memory/storage to request
- what labels/annotations identify this workload

For the workshop, the workload is tagged with your ACCESS username/project information so it is easier to connect the job to allocation/usage thinking.


In [ ]:
import hashlib
import sys

access_label = sanitize_label_value(ACCESS_USERNAME, fallback="access-user")
project_label = sanitize_label_value(ACCESS_PROJECT_ID, fallback="access-project")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
short_hash = hashlib.sha1(f"{ACCESS_USERNAME}-{timestamp}".encode()).hexdigest()[:6]
job_name = f"watersoft-cloud-{access_label[:20]}-{short_hash}"
job_name = sanitize_label_value(job_name, fallback="watersoft-cloud-job")

# Kubernetes accepts JSON manifests as well as YAML.
job_manifest = {
    "apiVersion": "batch/v1",
    "kind": "Job",
    "metadata": {
        "name": job_name,
        "labels": {
            "app": "watersofthack-cloud-job",
            "workshop": "watersofthack",
            "access-user": access_label,
            "access-project": project_label,
        },
        "annotations": {
            "watersofthack.org/access-username": ACCESS_USERNAME,
            "watersofthack.org/access-project-id": ACCESS_PROJECT_ID,
            "watersofthack.org/notebook": "02-access-credits-nautilus-job",
            "watersofthack.org/created-utc": datetime.now(timezone.utc).isoformat(),
        },
    },
    "spec": {
        "backoffLimit": 1,
        "ttlSecondsAfterFinished": 3600,
        "template": {
            "metadata": {
                "labels": {
                    "app": "watersofthack-cloud-job",
                    "workshop": "watersofthack",
                    "access-user": access_label,
                }
            },
            "spec": {
                "restartPolicy": "Never",
                "containers": [
                    {
                        "name": "water-summary",
                        "image": "python:3.11-slim",
                        "imagePullPolicy": "IfNotPresent",
                        "env": [
                            {"name": "ACCESS_USERNAME", "value": ACCESS_USERNAME},
                            {"name": "ACCESS_PROJECT_ID", "value": ACCESS_PROJECT_ID},
                            {"name": "WATER_REGION", "value": WATER_REGION},
                            {"name": "WORK_UNITS", "value": str(WORK_UNITS)},
                        ],
                        "command": ["python", "-c", job_python_code],
                        "resources": resource_plan,
                    }
                ],
            },
        },
    },
}

manifest_path = workspace / f"{job_name}.job.json"
manifest_path.write_text(json.dumps(job_manifest, indent=2), encoding="utf-8")

print("Job name:", job_name)
print("Manifest path:", manifest_path)
print(json.dumps(job_manifest, indent=2)[:2500] + "\n...")


# 11. Local manifest sanity checks

These checks do not contact Nautilus. They catch common mistakes before submission.


In [ ]:
errors = []

if placeholder(ACCESS_USERNAME):
    errors.append("ACCESS_USERNAME is still a placeholder.")
if placeholder(ACCESS_PROJECT_ID):
    errors.append("ACCESS_PROJECT_ID is still a placeholder.")
if placeholder(active_namespace):
    errors.append("NAUTILUS_NAMESPACE is still a placeholder.")

container = job_manifest["spec"]["template"]["spec"]["containers"][0]
if "nvidia.com/gpu" in json.dumps(container.get("resources", {})):
    errors.append("This workshop job should not request a GPU.")

req_cpu = parse_cpu_to_cores(container["resources"]["requests"]["cpu"])
lim_cpu = parse_cpu_to_cores(container["resources"]["limits"]["cpu"])
if lim_cpu / req_cpu > 1.21:
    errors.append("CPU limit is too far above CPU request for this workshop policy habit.")

req_mem = parse_memory_to_gib(container["resources"]["requests"]["memory"])
lim_mem = parse_memory_to_gib(container["resources"]["limits"]["memory"])
if lim_mem / req_mem > 1.21:
    errors.append("Memory limit is too far above memory request for this workshop policy habit.")

print("Sanity check errors:", len(errors))
for error in errors:
    print("-", error)

if not errors:
    print("Manifest looks ready for a small workshop run.")


# 12. Optional: ask the Kubernetes API to validate the manifest

This uses `kubectl apply --dry-run=server`, which checks the manifest against the cluster API without creating the job.

It may trigger login if your Nautilus token is expired.


In [ ]:
RUN_SERVER_DRY_RUN = False  # set True if the instructor wants everyone to validate against Nautilus

if RUN_SERVER_DRY_RUN:
    if not kubectl_path:
        print("kubectl is not installed in this environment.")
    elif placeholder(active_namespace):
        print("Set NAUTILUS_NAMESPACE before server dry-run.")
    else:
        run_command(["kubectl", "apply", "--dry-run=server", "-n", active_namespace, "-f", str(manifest_path)], timeout=120)
else:
    print("Server dry-run skipped. Set RUN_SERVER_DRY_RUN = True to validate against Nautilus.")


# 13. Submit the real Nautilus job

Only run this as a live job when your instructor says to do so.

To launch it:

1. make sure your ACCESS/project/namespace values are correct
2. set `RUN_REAL_NAUTILUS_JOB = True` in Cell 1
3. rerun Cell 1 and the cells below

The job is intentionally small and exits by itself. It does **not** use `sleep infinity` and it does **not** request a GPU.


In [ ]:
job_submitted = False

if RUN_REAL_NAUTILUS_JOB:
    if errors:
        print("Not submitting because manifest sanity checks found errors:")
        for error in errors:
            print("-", error)
    elif not kubectl_path:
        print("Not submitting because kubectl is not available.")
    elif placeholder(active_namespace):
        print("Not submitting because NAUTILUS_NAMESPACE is not set.")
    else:
        result = run_command(["kubectl", "apply", "-n", active_namespace, "-f", str(manifest_path)], timeout=120)
        job_submitted = result["ok"]
        print("job_submitted:", job_submitted)
else:
    print("Dry-run mode: no job submitted.")
    print("Generated manifest:", manifest_path)


# 14. Watch the job status

A Kubernetes Job creates one or more Pods.  
The Job controller watches the Pod until the command exits successfully or fails.


In [ ]:
if RUN_REAL_NAUTILUS_JOB and job_submitted:
    run_command(["kubectl", "get", "job", job_name, "-n", active_namespace, "-o", "wide"], timeout=60)
    run_command(["kubectl", "get", "pods", "-n", active_namespace, "-l", f"job-name={job_name}", "-o", "wide"], timeout=60)
else:
    print("No live job to watch in dry-run mode.")
    print("When live, the commands are:")
    print(f"kubectl get job {job_name} -n {active_namespace} -o wide")
    print(f"kubectl get pods -n {active_namespace} -l job-name={job_name} -o wide")


# 15. Wait for completion and read logs

The output printed by the container is the result of your remote cloud workload.


In [ ]:
job_logs = ""

if RUN_REAL_NAUTILUS_JOB and job_submitted:
    run_command(["kubectl", "wait", "--for=condition=complete", f"job/{job_name}", "-n", active_namespace, "--timeout=180s"], timeout=240)

    pod_lookup = run_command(["kubectl", "get", "pods", "-n", active_namespace, "-l", f"job-name={job_name}", "-o", "jsonpath={.items[0].metadata.name}"], timeout=60)
    pod_name = pod_lookup.get("stdout", "").strip()
    print("Pod name:", pod_name)

    if pod_name:
        logs_result = run_command(["kubectl", "logs", "-n", active_namespace, pod_name], timeout=120)
        job_logs = logs_result.get("stdout", "")
        logs_path = workspace / f"{job_name}.logs.txt"
        logs_path.write_text(job_logs, encoding="utf-8")
        print("Saved logs to:", logs_path)
else:
    print("No live job logs in dry-run mode.")


# 16. Parse the result from logs

When the real job runs, the logs contain a JSON block between markers.

This cell extracts that JSON block and saves it as a local result file.


In [ ]:
def extract_marked_json(text, start_marker, end_marker):
    if start_marker not in text or end_marker not in text:
        return None
    start = text.index(start_marker) + len(start_marker)
    end = text.index(end_marker, start)
    block = text[start:end].strip()
    return json.loads(block)

if job_logs:
    parsed = extract_marked_json(job_logs, "WATERSOFTHACK_JOB_RESULT_START", "WATERSOFTHACK_JOB_RESULT_END")
    if parsed:
        result_path = workspace / f"{job_name}.result.json"
        result_path.write_text(json.dumps(parsed, indent=2), encoding="utf-8")
        print("Saved parsed result:", result_path)
        print(json.dumps(parsed, indent=2))
    else:
        print("Could not find JSON result markers in logs.")
else:
    print("No logs available yet. Run the live job first to parse results.")


# 17. Inspect what resources the live job requested

This is the cloud/credit connection.

Even for a small job, the platform sees a formal request for CPU, memory, and ephemeral storage.


In [ ]:
if RUN_REAL_NAUTILUS_JOB and job_submitted:
    job_json_result = run_command(["kubectl", "get", "job", job_name, "-n", active_namespace, "-o", "json"], timeout=60)
    if job_json_result["ok"]:
        live_job = json.loads(job_json_result["stdout"])
        live_container = live_job["spec"]["template"]["spec"]["containers"][0]
        live_resources = live_container.get("resources", {})
        live_resources_path = workspace / f"{job_name}.live_resources.json"
        live_resources_path.write_text(json.dumps(live_resources, indent=2), encoding="utf-8")
        print(json.dumps(live_resources, indent=2))
        print("Saved:", live_resources_path)
else:
    print("Dry-run resource plan:")
    print(json.dumps(resource_plan, indent=2))


# 18. Clean up the job

Cleaning up is part of the exercise.

Cloud resources are shared. If you do not need a workload anymore, delete it.


In [ ]:
if RUN_REAL_NAUTILUS_JOB and job_submitted and DELETE_JOB_AFTER_LOGS:
    run_command(["kubectl", "delete", "job", job_name, "-n", active_namespace], timeout=120)
    print("Cleanup requested for job:", job_name)
elif RUN_REAL_NAUTILUS_JOB and job_submitted:
    print("DELETE_JOB_AFTER_LOGS is False, so the job was not deleted by this notebook.")
    print("Manual cleanup command:")
    print(f"kubectl delete job {job_name} -n {active_namespace}")
else:
    print("No live job to clean up in dry-run mode.")


# 19. What to check in ACCESS / allocation portal after the run

This notebook cannot reliably query your ACCESS credit balance from inside the notebook environment.

After the live job, manually check your allocation/usage dashboard:

1. log into the ACCESS allocations portal
2. open the project used for this workshop
3. go to the **Credits + Resources** area
4. look for resource usage details
5. compare the resource/accounting view with the job metadata you saved here

This is the cloud habit: connect the code you ran to the resources and credits behind it.


In [ ]:
usage_checklist = {
    "access_username": ACCESS_USERNAME,
    "access_project_id": ACCESS_PROJECT_ID,
    "nautilus_namespace": active_namespace,
    "job_name": job_name,
    "manifest_path": str(manifest_path),
    "resource_plan_path": str(resource_plan_path),
    "planned_footprint_path": str(footprint_path),
    "what_to_check_manually": [
        "ACCESS allocations portal project page",
        "Credits + Resources tab",
        "Usage details icon/table",
        "Nautilus namespace/job labels if available in dashboards",
    ],
}

usage_checklist_path = workspace / "manual_access_usage_checklist.json"
usage_checklist_path.write_text(json.dumps(usage_checklist, indent=2), encoding="utf-8")
print(usage_checklist_path.read_text(encoding="utf-8"))


# 20. Optional challenge: change one thing and rerun

Only do this if your instructor says the class has enough time and allocation budget.

Possible safe changes:

- increase `WORK_UNITS` from `25_000` to `50_000`
- change `WATER_REGION`
- compare runtime seconds in the JSON logs

Do **not** request GPU for this notebook.  
Do **not** run hundreds of jobs.  
Do **not** leave jobs around after collecting logs.


# 21. Reflection prompt

Write a short answer in a new markdown cell:

1. Which identity/project/namespace values connected your work to Nautilus and ACCESS?
2. What resources did the job request?
3. Where would you check official usage or credit information?
4. Why is cleanup part of cloud computing practice?


# 22. Final checklist

You are done with this notebook if you have completed the following:

- [ ] filled in ACCESS username / project / Nautilus namespace
- [ ] checked whether `kubectl` is available
- [ ] identified the current Kubernetes context and namespace
- [ ] created `workshop_identity_card.json`
- [ ] created `resource_plan.json`
- [ ] tested the workload code locally
- [ ] generated the Kubernetes Job manifest
- [ ] ran the server dry-run or reviewed the manifest with the instructor
- [ ] submitted the real Nautilus job, if instructed
- [ ] read job status and logs, if instructed
- [ ] saved or parsed the result logs, if instructed
- [ ] cleaned up the job
- [ ] checked where usage/credits are tracked outside the notebook

---

## Done

This notebook used Nautilus as a real cloud platform: identity, namespace, resource request, job execution, logs, usage thinking, and cleanup.


# References used by the instructor

These are here so students can connect the workshop to official documentation:

- NRP Nautilus Getting Started: https://nrp.ai/documentation/userdocs/start/getting-started/
- NRP Nautilus JupyterHub Service: https://nrp.ai/documentation/userdocs/jupyter/jupyterhub-service/
- NRP Nautilus Running Batch Jobs: https://nrp.ai/documentation/userdocs/running/jobs/
- NRP Nautilus Cluster Policies: https://nrp.ai/documentation/userdocs/start/policies/
- ACCESS Allocations How-To: https://allocations.access-ci.org/how-to
